In [1]:
# Imports and config

import os, json
import numpy as np
from PIL import Image
from tqdm import tqdm
import skimage.draw
import multiprocessing as mp

# ── PATHS ─────────────────────────────────────────────
DATA_ROOT = '/kaggle/input/datasets/varun000reddy/'

VAL_IMG = DATA_ROOT + 'validation/validation/image/'
VAL_ANN = DATA_ROOT + 'validation/validation/annos/'

SAVE_ROOT = '/kaggle/working/processed_val/'
SAVE_IMG  = SAVE_ROOT + 'images/'
SAVE_MASK = SAVE_ROOT + 'masks/'

IMG_SIZE = 512

TOP5_CATEGORIES = {1: 1, 8: 2, 7: 3, 2: 4, 9: 5}

os.makedirs(SAVE_IMG, exist_ok=True)
os.makedirs(SAVE_MASK, exist_ok=True)

print("Setup complete")

Setup complete


In [2]:
# Load annotation files

ann_files = sorted([f for f in os.listdir(VAL_ANN) if f.endswith('.json')])

print(f"Total validation annotation files: {len(ann_files)}")

Total validation annotation files: 32153


In [3]:
# Processing Function

def process_val_file(fname):
    try:
        img_id   = fname.replace('.json', '')
        img_path = os.path.join(VAL_IMG, img_id + '.jpg')
        ann_path = os.path.join(VAL_ANN, fname)

        if not os.path.exists(img_path):
            return None

        mask_path = os.path.join(SAVE_MASK, img_id + '.png')
        img_save_path = os.path.join(SAVE_IMG, img_id + '.jpg')

        # Skip already processed
        if os.path.exists(mask_path):
            return img_id

        # Load image
        img = Image.open(img_path).convert('RGB')
        orig_w, orig_h = img.size

        img_resized = img.resize((IMG_SIZE, IMG_SIZE))
        img_resized.save(img_save_path)

        scale_x = IMG_SIZE / orig_w
        scale_y = IMG_SIZE / orig_h

        with open(ann_path) as f:
            ann = json.load(f)

        mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)
        has_valid_object = False

        for key, item in ann.items():
            if not key.startswith('item'):
                continue

            cat_id = item.get('category_id')
            if cat_id not in TOP5_CATEGORIES:
                continue

            class_idx = TOP5_CATEGORIES[cat_id]
            segs      = item.get('segmentation', [])

            for polygon in segs:
                if len(polygon) < 6:
                    continue

                px = np.array(polygon[0::2], dtype=np.float32) * scale_x
                py = np.array(polygon[1::2], dtype=np.float32) * scale_y

                px = np.clip(px, 0, IMG_SIZE - 1)
                py = np.clip(py, 0, IMG_SIZE - 1)

                rr, cc = skimage.draw.polygon(py, px,
                         shape=(IMG_SIZE, IMG_SIZE))

                mask[rr, cc] = class_idx
                has_valid_object = True

        if not has_valid_object:
            return None

        Image.fromarray(mask).save(mask_path)

        return img_id

    except:
        return None

In [4]:
# Multiprocessing

print(" Precomputing validation masks (parallel)...")

NUM_WORKERS = 4  # optimal for Kaggle

with mp.Pool(NUM_WORKERS) as pool:
    results = list(tqdm(pool.imap(process_val_file, ann_files),
                        total=len(ann_files)))

valid_ids = [r for r in results if r is not None]

print(f"\n Validation masks created for {len(valid_ids)} images")

 Precomputing validation masks (parallel)...


100%|██████████| 32153/32153 [11:46<00:00, 45.52it/s]


 Validation masks created for 23741 images


In [5]:
# Save Valid Ids

with open(os.path.join(SAVE_ROOT, 'valid_ids.json'), 'w') as f:
    json.dump(valid_ids, f)

print(" Saved validation valid_ids.json")

print("Images saved :", len(os.listdir(SAVE_IMG)))
print("Masks saved  :", len(os.listdir(SAVE_MASK)))

 Saved validation valid_ids.json
Images saved : 32153
Masks saved  : 23741


In [6]:
# Upload the validation Dataset

import shutil, subprocess

UPLOAD_DIR = '/kaggle/working/processed_val_upload/'
os.makedirs(UPLOAD_DIR, exist_ok=True)

shutil.copytree(SAVE_IMG,  UPLOAD_DIR + '/images', dirs_exist_ok=True)
shutil.copytree(SAVE_MASK, UPLOAD_DIR + '/masks',  dirs_exist_ok=True)

metadata = {
    "title": "vr-processed-val-segmentation",
    "id": "pankajdeopaiiitb/vr-processed-val-segmentation",
    "licenses": [{"name": "CC0-1.0"}]
}

with open(UPLOAD_DIR + '/dataset-metadata.json', 'w') as f:
    json.dump(metadata, f)

result = subprocess.run(
    ['kaggle', 'datasets', 'create', '-p', UPLOAD_DIR, '--dir-mode', 'zip'],
    capture_output=True, text=True
)

print(result.stdout)
print(result.stderr)

Starting upload for file masks.zip
Upload successful: masks.zip (44MB)
Starting upload for file images.zip
Upload successful: images.zip (1012MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/pankajdeopaiiitb/vr-processed-val-segmentation


  0%|          | 0.00/44.1M [00:00<?, ?B/s]
  4%|▎         | 1.55M/44.1M [00:00<00:14, 3.02MB/s]
  5%|▍         | 2.03M/44.1M [00:00<00:13, 3.25MB/s]
  8%|▊         | 3.45M/44.1M [00:00<00:08, 5.31MB/s]
 12%|█▏        | 5.42M/44.1M [00:00<00:05, 8.10MB/s]
 18%|█▊        | 7.81M/44.1M [00:01<00:03, 12.0MB/s]
 25%|██▌       | 11.2M/44.1M [00:01<00:02, 15.9MB/s]
 32%|███▏      | 14.3M/44.1M [00:01<00:01, 19.7MB/s]
 44%|████▎     | 19.2M/44.1M [00:01<00:00, 27.0MB/s]
 55%|█████▍    | 24.2M/44.1M [00:01<00:00, 31.0MB/s]
 66%|██████▌   | 29.0M/44.1M [00:01<00:00, 35.7MB/s]
 76%|███████▌  | 33.5M/44.1M [00:01<00:00, 38.8MB/s]
 87%|████████▋ | 38.2M/44.1M [00:01<00:00, 41.5MB/s]
 98%|█████████▊| 43.2M/44.1M 